# Versioned result format and handling

Akira stores an inference result as a human-readable JSON summary plus an NPZ archive of numerical arrays. The explicit format version lets future Akira releases recognize, migrate, or reject old files rather than silently misinterpret them.

This tutorial fits a compact example, saves it, inspects both parts, reloads it with validation, and uses the loaded arrays without requiring the original live result object.

In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import numpy as np

from akira import (
    CartesianGrid, CompositeLight, FreeParameter, Gaussian, InferenceProblem,
    LensModel, LensPlane, ProfileModel, RESULT_FORMAT_VERSION, SIS,
    UniformPrior, fit_model, gaussian_psf, load_inference_result, render,
    save_inference_result, simulate_imaging,
)

## 1. Create and fit a small inference problem

The result format is independent of model complexity. Here only the SIS Einstein radius is free, which keeps the tutorial fast while exercising the same storage path as a larger analytic model.

In [ ]:
grid = CartesianGrid((32, 32), pixel_scale=0.07, supersampling=2)
true_lens = LensPlane((SIS(0.78),))
true_source = CompositeLight((
    Gaussian(0.9, 0.13, centre=(0.05, -0.025), q=0.8, phi=0.2),
))
image = render(grid, true_lens, true_source)
observation = simulate_imaging(
    image, gaussian_psf(0.7, size=5), noise_sigma=0.008,
    pixel_scale=grid.pixel_scale, seed=41,
)
model = LensModel(
    lens=(ProfileModel(
        SIS, einstein_radius=FreeParameter(0.72, UniformPrior(0.5, 1.0))
    ),),
    source=(ProfileModel(
        Gaussian, intensity=0.9, sigma=0.13, centre=(0.05, -0.025),
        q=0.8, phi=0.2,
    ),),
)
problem = InferenceProblem(model, observation, grid)
result = fit_model(model, observation, grid, mode='map')
print(result.success, result.parameters)

## 2. Save JSON metadata and NPZ arrays

For this demonstration a temporary directory avoids leaving generated files beside the notebook. In a real analysis, pass a durable path such as `outputs/lens_001/map_fit`. The writer returns both created paths.

In [ ]:
temporary_directory = TemporaryDirectory()
output_directory = Path(temporary_directory.name)
summary_path, arrays_path = save_inference_result(
    output_directory, problem, result
)
print(summary_path.name, f'({summary_path.stat().st_size:,} bytes)')
print(arrays_path.name, f'({arrays_path.stat().st_size:,} bytes)')

## 3. Inspect the human-readable summary

The summary records provenance, the complete problem definition, fixed and free profile values, flattened parameter ordering, point estimate, optimizer diagnostics, and a manifest for every stored array. The `sampling` field is currently `null`; NUTS and nested-sampling results will populate it later.

In [ ]:
summary = json.loads(summary_path.read_text())
print('format:', summary['format'])
print('format version:', summary['format_version'])
print('Akira version:', summary['provenance']['akira_version'])
print('objective:', summary['provenance']['objective'])
print('model profiles:', [p['type'] for p in summary['problem']['model']['lens']])
print('point estimate:', summary['point_estimate']['parameters'])
print('array keys:', list(summary['arrays']))
assert summary['format_version'] == RESULT_FORMAT_VERSION

In [ ]:
# A representative free-parameter declaration preserved in JSON.
print(json.dumps(summary['problem']['model']['lens'][0], indent=2))

## 4. Load through Akira's validated reader

Use `load_inference_result` instead of opening the NPZ archive directly. It checks the format identifier and version and verifies each array key, shape, and data type against the JSON manifest. Pickled objects are disabled.

The returned `StoredInferenceResult` deliberately contains portable dictionaries and NumPy arrays rather than attempting to recreate arbitrary live Python objects.

In [ ]:
stored = load_inference_result(output_directory)
for name, array in stored.arrays.items():
    print(f'{name:18s} shape={str(array.shape):12s} dtype={array.dtype}')

assert np.allclose(stored.arrays['model_image'], np.asarray(result.model_image))
assert np.allclose(
    stored.arrays['residual_map'],
    stored.arrays['observation_data'] - stored.arrays['model_image'],
)

## 5. Work with a loaded result

The archive contains enough information for ordinary inspection and plotting even after the original optimization process has ended.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), constrained_layout=True)
panels = (
    ('observation_data', 'Observed image'),
    ('model_image', 'MAP model'),
    ('residual_map', 'Residual'),
)
for axis, (key, title) in zip(axes, panels):
    artist = axis.imshow(stored.arrays[key], origin='lower', cmap='magma')
    axis.set_title(title)
    axis.set_xticks([]); axis.set_yticks([])
    fig.colorbar(artist, ax=axis, fraction=0.046)
plt.show()

temporary_directory.cleanup()

## Current scope

Format version 1 is an archival analytic-optimization result, not a restart checkpoint. It does not resume an optimizer or reconstruct arbitrary Python objects. Planned posterior sampling will retain this top-level structure while adding chain arrays, posterior summaries, convergence diagnostics, weights, and evidence where appropriate.